# Federation market walkthrough

## 1. The federation

The fixture runs on a 10 percent share of the four default GPU platforms.

In [1]:
import os
from pathlib import Path
import shutil
import subprocess
import sys

from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "CMakeLists.txt").exists():
    ROOT = ROOT.parent
if not (ROOT / "CMakeLists.txt").exists():
    raise RuntimeError("not inside a dr_evt checkout")
sys.path[:0] = [
    str(ROOT / "install" / "lib" / "python"),
    str(ROOT / "python"),
]

from dr_evt_market import (
    FirstPrice,
    Vcg,
    candidates,
    federation,
    read_jobs,
    run,
    write_outputs,
)

LEARN = ROOT / "python" / "dr_evt_market" / "learn"
WORK = LEARN / "output" / "fixture"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)
DATA = ROOT / "python" / "dr_evt_market" / "tests" / "data"

def _table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |"]
    lines.append("|" + "|".join("---" for _ in headers) + "|")
    lines.extend("| " + " | ".join(map(str, row)) + " |" for row in rows)
    return "\n".join(lines)

platforms = federation(WORK / "hand-platforms", share=0.1)
platform_rows = [
    (
        name,
        platform.total_nodes,
        platform.exposed_nodes,
        platform.price_per_node_hour,
        " ".join(sorted(platform.hardware)),
    )
    for name, platform in platforms.items()
]
display(
    Markdown(
        _table(
            ("name", "total", "exposed", "price per node-hour", "hardware"),
            platform_rows,
        )
    )
)

| name | total | exposed | price per node-hour | hardware |
|---|---|---|---|---|
| corona | 121 | 12 | 2.0 | amd cpu gpu |
| lassen | 795 | 80 | 3.0 | cpu gpu nvidia |
| tioga | 32 | 3 | 6.0 | amd cpu gpu |
| tuolumne | 1152 | 115 | 8.0 | amd cpu gpu |

## 2. The stream and its bids

A bid is the most the user will pay per node-hour. It is either one price for
every compatible platform or a mapping with one price for each selected
platform. The execution limit is the historical run time, with the original
request carried beside it.

In [2]:
jobs = read_jobs(DATA / "jobs.csv")

def _bid(job):
    if isinstance(job.bid, dict):
        return " ".join(f"{name}:{price:g}" for name, price in job.bid.items())
    return f"{job.bid:g}"

job_rows = [
    (
        job.job_id,
        job.submit_s,
        job.num_nodes,
        job.limit_s,
        job.requested_s,
        _bid(job),
        " ".join(sorted(job.requires)),
        job.source,
        job.persona,
    )
    for job in jobs
]
display(
    Markdown(
        _table(
            (
                "job",
                "submit",
                "nodes",
                "limit",
                "requested",
                "bid",
                "requires",
                "source",
                "persona",
            ),
            job_rows,
        )
    )
)

| job | submit | nodes | limit | requested | bid | requires | source | persona |
|---|---|---|---|---|---|---|---|---|
| j000001 | 0 | 8 | 90 | 120 | 2.2 | gpu | tioga | tier |
| j000002 | 0 | 7 | 150 | 180 | 2.4 | gpu | tuo | value |
| j000003 | 0 | 6 | 55 | 60 | 3.2 | gpu | corona | tier |
| j000004 | 0 | 20 | 100 | 120 | lassen:3.2 tuolumne:8.5 | gpu nvidia | tioga | tier |
| j000005 | 0 | 3 | 80 | 90 | 2.6 | amd gpu | tuo | value |
| j000006 | 30 | 10 | 70 | 90 | 2 | gpu | corona | sticker |
| j000007 | 60 | 12 | 100 | 120 | 6.5 | gpu | tioga | value |
| j000008 | 60 | 3 | 50 | 60 | 8 | amd gpu | tuo | tier |
| j000009 | 90 | 30 | 160 | 180 | 3.5 | gpu nvidia | corona | tier |
| j000010 | 90 | 2 | 40 | 45 | corona:20 lassen:12 tioga:10 tuolumne:9 | gpu | tioga | whale |
| j000011 | 120 | 11 | 55 | 60 | 4 | gpu | tuo | tier |
| j000012 | 120 | 3 | 75 | 90 | 2.5 | amd gpu | corona | tier |
| j000013 | 150 | 9 | 100 | 120 | 7 | gpu | tioga | value |
| j000014 | 180 | 120 | 80 | 90 | 6 | gpu | tuo | value |
| j000015 | 180 | 4 | 50 | 60 | corona:1.5 | gpu | corona | sticker |
| j000016 | 210 | 14 | 105 | 120 | 4 | gpu | tioga | tier |
| j000017 | 240 | 3 | 50 | 60 | 8.5 | amd gpu | tuo | tier |
| j000018 | 240 | 18 | 75 | 90 | 4 | gpu nvidia | corona | tier |
| j000019 | 270 | 6 | 40 | 45 | 6.5 | gpu | tioga | value |
| j000020 | 300 | 5 | 25 | 30 | 8.5 | gpu | tuo | value |

## 3. One window by hand

At time zero, candidacy combines hardware, free capacity, posted prices, and
the bid before VCG chooses an allocation.

In [3]:
for platform in platforms.values():
    platform.advance_to(0)
first_batch = [job for job in jobs if job.submit_s <= 0]
free = {name: platform.free_nodes() for name, platform in platforms.items()}
offers = {
    job.job_id: candidates(job, platforms, free) for job in first_batch
}
candidate_rows = []
for job in first_batch:
    for name, (cost, value) in offers[job.job_id].items():
        candidate_rows.append(
            (
                job.job_id,
                name,
                f"{platforms[name].price_per_node_hour:.1f}",
                f"{job.price(name):.1f}",
                f"{cost:.4f}",
                f"{value:.4f}",
                f"{value - cost:.4f}",
            )
        )
display(Markdown("### Candidates"))
display(
    Markdown(
        _table(
            ("job", "platform", "posted", "bid", "cost", "value", "net"),
            candidate_rows,
        )
    )
)
decisions = Vcg().decide(first_batch, platforms, free)
decision_rows = []
for decision in decisions:
    cost, value = offers[decision.job_id][decision.platform]
    decision_rows.append(
        (
            decision.job_id,
            decision.platform,
            f"{cost:.4f}",
            f"{decision.charge - cost:.4f}",
            f"{decision.charge:.4f}",
        )
    )
display(Markdown("### VCG decisions"))
display(
    Markdown(
        _table(
            ("job", "platform", "cost", "premium", "charge"),
            decision_rows,
        )
    )
)
display(
    Markdown(
        "A premium is the savings displaced jobs would have had if the "
        "winner were absent."
    )
)

### Candidates

| job | platform | posted | bid | cost | value | net |
|---|---|---|---|---|---|---|
| j000001 | corona | 2.0 | 2.2 | 0.4000 | 0.4400 | 0.0400 |
| j000002 | corona | 2.0 | 2.4 | 0.5833 | 0.7000 | 0.1167 |
| j000003 | corona | 2.0 | 3.2 | 0.1833 | 0.2933 | 0.1100 |
| j000003 | lassen | 3.0 | 3.2 | 0.2750 | 0.2933 | 0.0183 |
| j000004 | lassen | 3.0 | 3.2 | 1.6667 | 1.7778 | 0.1111 |
| j000005 | corona | 2.0 | 2.6 | 0.1333 | 0.1733 | 0.0400 |

### VCG decisions

| job | platform | cost | premium | charge |
|---|---|---|---|---|
| j000002 | corona | 0.5833 | 0.0917 | 0.6750 |
| j000003 | lassen | 0.2750 | 0.0000 | 0.2750 |
| j000004 | lassen | 1.6667 | 0.0000 | 1.6667 |
| j000005 | corona | 0.1333 | 0.0000 | 0.1333 |

A premium is the savings displaced jobs would have had if the winner were absent.

## 4. Complete runs

The fixture is routed first by VCG and then by pay what you bid. The same
costs and reported values make their welfare and revenue directly comparable.

In [4]:
def _run_one(mechanism):
    name = mechanism.name
    current = federation(WORK / name / "platforms", share=0.1)
    result = run(jobs, current, mechanism, window_s=60, prefix=32)
    paths = write_outputs(result, WORK / name / "outputs")
    return result, paths

vcg_result, vcg_paths = _run_one(Vcg())
routed_rows = [
    (
        row.job_id,
        row.platform,
        row.window,
        row.begin_s,
        row.end_s,
        f"{row.cost:.4f}",
        f"{row.value:.4f}",
        f"{row.premium:.4f}",
        f"{row.charge:.4f}",
    )
    for row in vcg_result.routed
]
display(Markdown("### VCG routed jobs"))
display(
    Markdown(
        _table(
            (
                "job",
                "platform",
                "window",
                "begin",
                "end",
                "cost",
                "value",
                "premium",
                "charge",
            ),
            routed_rows,
        )
    )
)
rejected_rows = [
    (row.job_id, row.reason, row.time_s) for row in vcg_result.rejected
]
display(Markdown("### Rejected jobs"))
display(Markdown(_table(("job", "reason", "time"), rejected_rows)))
stat_fields = (
    "jobs_submitted",
    "jobs_completed",
    "avg_wait_time",
    "utilization",
    "makespan",
)
stat_rows = [
    (name, *(f"{values[field]:.4f}" for field in stat_fields))
    for name, values in vcg_result.statistics.items()
]
display(Markdown("### VCG platform statistics"))
display(Markdown(_table(("platform", *stat_fields), stat_rows)))
print(f"routed_sha256={vcg_paths['sha256']}")

first_result, _ = _run_one(FirstPrice())
comparison = []
for name, result in (("vcg", vcg_result), ("firstprice", first_result)):
    comparison.append(
        (
            name,
            len(result.routed),
            len(result.rejected),
            f"{sum(row.value - row.cost for row in result.routed):.4f}",
            f"{sum(row.charge for row in result.routed):.4f}",
        )
    )
display(Markdown("### Mechanism comparison"))
display(
    Markdown(
        _table(
            ("mechanism", "routed", "rejected", "welfare", "revenue"),
            comparison,
        )
    )
)

### VCG routed jobs

| job | platform | window | begin | end | cost | value | premium | charge |
|---|---|---|---|---|---|---|---|---|
| j000002 | corona | 0 | 0 | 150 | 0.5833 | 0.7000 | 0.0917 | 0.6750 |
| j000003 | lassen | 0 | 0 | 55 | 0.2750 | 0.2933 | 0.0000 | 0.2750 |
| j000004 | lassen | 0 | 0 | 100 | 1.6667 | 1.7778 | 0.0000 | 1.6667 |
| j000005 | corona | 0 | 0 | 80 | 0.1333 | 0.1733 | 0.0000 | 0.1333 |
| j000007 | lassen | 1 | 60 | 160 | 1.0000 | 2.1667 | 0.0000 | 1.0000 |
| j000008 | tioga | 1 | 60 | 110 | 0.2500 | 0.3333 | 0.0000 | 0.2500 |
| j000009 | lassen | 2 | 120 | 280 | 4.0000 | 4.6667 | 0.0000 | 4.0000 |
| j000010 | corona | 2 | 120 | 160 | 0.0444 | 0.4444 | 0.0000 | 0.0444 |
| j000011 | lassen | 2 | 120 | 175 | 0.5042 | 0.6722 | 0.0000 | 0.5042 |
| j000012 | corona | 2 | 120 | 195 | 0.1250 | 0.1562 | 0.0000 | 0.1250 |
| j000013 | corona | 3 | 180 | 280 | 0.5000 | 1.7500 | 0.0400 | 0.5400 |
| j000016 | lassen | 4 | 240 | 345 | 1.2250 | 1.6333 | 0.0000 | 1.2250 |
| j000017 | corona | 4 | 240 | 290 | 0.0833 | 0.3542 | 0.0000 | 0.0833 |
| j000018 | lassen | 4 | 240 | 315 | 1.1250 | 1.5000 | 0.0000 | 1.1250 |
| j000019 | corona | 5 | 300 | 340 | 0.1333 | 0.4333 | 0.0053 | 0.1386 |
| j000020 | corona | 5 | 300 | 325 | 0.0694 | 0.2951 | 0.0000 | 0.0694 |
| j000001 | corona | 6 | 360 | 450 | 0.4000 | 0.4400 | 0.0000 | 0.4000 |
| j000006 | corona | 8 | 480 | 550 | 0.3889 | 0.3889 | 0.0000 | 0.3889 |

### Rejected jobs

| job | reason | time |
|---|---|---|
| j000014 | oversize | 180 |
| j000015 | unaffordable | 180 |

### VCG platform statistics

| platform | jobs_submitted | jobs_completed | avg_wait_time | utilization | makespan |
|---|---|---|---|---|---|
| corona | 10.0000 | 10.0000 | 0.0000 | 0.6712 | 550.0000 |
| lassen | 7.0000 | 7.0000 | 0.0000 | 0.4259 | 345.0000 |
| tioga | 1.0000 | 1.0000 | 0.0000 | 0.4545 | 110.0000 |
| tuolumne | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000 |

routed_sha256=3cca4e4cce7106a213753c9fff762c62a354b9e680b66195fef59465f51c4ae9


### Mechanism comparison

| mechanism | routed | rejected | welfare | revenue |
|---|---|---|---|---|
| vcg | 18 | 2 | 5.6719 | 12.6439 |
| firstprice | 18 | 2 | 5.6719 | 18.1789 |

## 5. Command line

The package exposes one command for a prepared stream and one for trace
preparation. Both examples below use the fixture data.

In [5]:
environment = os.environ.copy()
environment["PYTHONPATH"] = os.pathsep.join(
    (str(ROOT / "install" / "lib" / "python"), str(ROOT / "python"))
)
commands = [
    [
        sys.executable,
        "-m",
        "dr_evt_market",
        "run",
        "--jobs",
        str(DATA / "jobs.csv"),
        "--out",
        str(WORK / "command-run"),
        "--share",
        "0.1",
    ],
    [
        sys.executable,
        "-m",
        "dr_evt_market",
        "prepare",
        "--trace",
        f"corona={DATA / 'trace.csv'}",
        "--trace",
        f"tioga={DATA / 'trace.csv'}",
        "--out",
        str(WORK / "command-jobs.csv"),
        "--start",
        "1000",
        "--hours",
        "0.05",
    ],
]
for command in commands:
    shown = " ".join(command[1:]).replace(f"{ROOT}/", "")
    print("$ python", shown)
    completed = subprocess.run(
        command,
        cwd=ROOT,
        env=environment,
        check=True,
        capture_output=True,
        text=True,
    )
    print(completed.stdout.strip())

$ python -m dr_evt_market run --jobs python/dr_evt_market/tests/data/jobs.csv --out python/dr_evt_market/learn/output/fixture/command-run --share 0.1


windows=9
routed=18
rejected=2
routed_sha256=3cca4e4cce7106a213753c9fff762c62a354b9e680b66195fef59465f51c4ae9
$ python -m dr_evt_market prepare --trace corona=python/dr_evt_market/tests/data/trace.csv --trace tioga=python/dr_evt_market/tests/data/trace.csv --out python/dr_evt_market/learn/output/fixture/command-jobs.csv --start 1000 --hours 0.05
read=24
kept=10
no_nodes=2
no_limit=2
no_start=4
bad_runtime=2
outside_interval=4
kept:corona=5
kept:tioga=5
